# Certified separation and admissible transformations

Exact rational exploration of the corrected R12/R13 results. The positive
Hermitian form and readout are supplied. Finite pockets test concrete
examples; they do not exhaust the rational ray space or establish its
classification. This notebook uses the paper's Python reference model,
not a Rust binding or an extraction of the Lean proofs. Authorization is
independently supplied. No model calls are made.

In [1]:
from pathlib import Path
import sys
from fractions import Fraction

root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'code' / 'certified_preservers').is_dir()), None)
if root is None:
    raise RuntimeError('Run from the paper root or jupyter_notebooks directory')
sys.path.insert(0, str(root / 'code'))
from certified_preservers.checker import check
from certified_preservers.examples import (
    graph_pocket, hadamard, qubit_countermodel, reference_problems,
)
from certified_preservers.model import G, I, scale
from certified_preservers.propose import propose
from certified_preservers.replay import replay
from state_effect_readout.model import identity, matmul

def require(condition, explanation):
    if not condition:
        raise RuntimeError(explanation)

problems = reference_problems()
print('Exact Q(i) reference ready; independently supplied inputs.')

Exact Q(i) reference ready; independently supplied inputs.


In [2]:
phase = G.of(Fraction(3, 5), Fraction(4, 5))
require(phase.norm() == 1, 'phase norm failed')
require(phase not in (G.of(1), G.of(-1), I, -I), 'counterexample failed')
print('Phase (3+4i)/5 has norm 1 and is outside the four Gaussian-integer units.')
print('Its square:', phase * phase)

Phase (3+4i)/5 has norm 1 and is outside the four Gaussian-integer units.
Its square: GaussianRational(re=Fraction(-7, 25), im=Fraction(24, 25))


In [3]:
candidate = propose(problems['hadamard'])['certificate']
result = check(problems['hadamard'], candidate)
require(result.status == 'VERIFIED', result.reason)
require(matmul(hadamard(), hadamard()) == scale(I, identity(2)), 'Hadamard square failed')
print(result.status, result.claim)
print('Gaussian Hadamard: U*U = I, U^2 = iI.')

VERIFIED global_semisimilarity
Gaussian Hadamard: U*U = I, U^2 = iI.


In [4]:
for name in (
    'reachable', 'norm_obstruction', 'luders', 'zero_branch', 'nonstandard_gram',
):
    certificate = propose(problems[name])['certificate']
    result = check(problems[name], certificate)
    require(result.status == 'VERIFIED', result.reason)
    print(name, result.claim)
    if name == 'norm_obstruction':
        print('Odd-valuation witness:', certificate['evidence'])
    if name in ('luders', 'zero_branch'):
        print('Probability:', certificate['evidence']['probability'])

reachable unitary_reachable
norm_obstruction unitary_impossible
Odd-valuation witness: {'prime': 3, 'valuation': -1}
luders positive_postselection
Probability: 1/3
zero_branch zero_branch
Probability: 0
nonstandard_gram unitary_reachable


In [5]:
certificate = propose(problems['even_similarity'])['certificate']
result = check(problems['even_similarity'], certificate)
require(result.status == 'VERIFIED', result.reason)
require(certificate['evidence']['multiplier'] == '3', 'wrong similarity multiplier')
print('Dimension 4: M*M = 3I; all ray probabilities are preserved.')
print('The odd 3-adic valuation prevents unitary normalization over Q(i).')

Dimension 4: M*M = 3I; all ray probabilities are preserved.
The odd 3-adic valuation prevents unitary normalization over Q(i).


In [6]:
countermodel = qubit_countermodel()
require(countermodel['before'] == '1/2' and countermodel['after'] == '4/5', 'boundary failed')
print('Orthogonality-preserving permutation on six complement-closed rays.')
print('Probability changes from', countermodel['before'], 'to', countermodel['after'])

Orthogonality-preserving permutation on six complement-closed rays.
Probability changes from 1/2 to 4/5


In [7]:
for dimension in (3, 4):
    pocket = graph_pocket(dimension)
    print({'dimension': dimension, 'rays': len(pocket['rays']),
           'orthogonal_pairs': len(pocket['orthogonal_pairs']),
           'probability_comparisons': pocket['probability_comparisons']})
print('Only these declared finite pockets were enumerated.')

{'dimension': 3, 'rays': 9, 'orthogonal_pairs': 9, 'probability_comparisons': 36}
{'dimension': 4, 'rays': 16, 'orthogonal_pairs': 42, 'probability_comparisons': 120}
Only these declared finite pockets were enumerated.


In [8]:
certificate = propose(problems['generator'])['certificate']
result = check(problems['generator'], certificate)
require(result.status == 'VERIFIED', result.reason)
print('Matrix-unit table:', result.claim)
print('No time evolution or matrix exponential has been inferred.')
print(replay(root / 'code' / 'certified_preservers' / 'replay_examples.json'))

Matrix-unit table: inner_dagger_derivation
No time evolution or matrix exponential has been inferred.
{'status': 'VERIFIED', 'episodes': 8, 'pockets': 2, 'model_calls': 0}
